In [1]:
import torch
torch.cuda.empty_cache()
import torch.nn as nn
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

In [2]:
class ContractiveAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ContractiveAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(output_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
            nn.Sigmoid()
        )
        self.criterion = nn.MSELoss()
        self.beta = 1e-4

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

    def loss_function(self, x, decoded, encoded):
        mse_loss = self.criterion(decoded, x)
        jacobian = torch.autograd.functional.jacobian(lambda x: encoded, x)
        frobenius_norm = torch.norm(jacobian, p='fro')
        contraction_loss = torch.mean(frobenius_norm)
        total_loss = mse_loss + (self.beta * contraction_loss)
        return total_loss

In [3]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
# Hyperparameters
input_size = 784  # MNIST images are 28x28 pixels
hidden_size = 128
output_size = 64
learning_rate = 0.001
num_epochs = 10
batch_size = 32


In [5]:
# Load MNIST dataset
train_dataset = MNIST(root="./data", train=True, transform=ToTensor(), download=True)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
# Initialize the autoencoder
autoencoder = ContractiveAutoencoder(input_size, hidden_size, output_size).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)

In [7]:
# Training loop
total_steps = len(train_dataloader)
for epoch in range(num_epochs):
    for i, (images, _) in enumerate(train_dataloader):
        # Move images to the device
        images = images.to(device)
        inputs = images.view(images.size(0), -1)

        # Forward pass
        encoded, decoded = autoencoder(inputs)

        # Compute the loss
        loss = autoencoder.loss_function(inputs, decoded, encoded)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_steps}], Loss: {loss.item():.4f}")

Epoch [1/10], Step [50/1875], Loss: 0.0755
Epoch [1/10], Step [100/1875], Loss: 0.0647
Epoch [1/10], Step [150/1875], Loss: 0.0600
Epoch [1/10], Step [200/1875], Loss: 0.0470
Epoch [1/10], Step [250/1875], Loss: 0.0467
Epoch [1/10], Step [300/1875], Loss: 0.0383
Epoch [1/10], Step [350/1875], Loss: 0.0360
Epoch [1/10], Step [400/1875], Loss: 0.0367
Epoch [1/10], Step [450/1875], Loss: 0.0367
Epoch [1/10], Step [500/1875], Loss: 0.0300
Epoch [1/10], Step [550/1875], Loss: 0.0336
Epoch [1/10], Step [600/1875], Loss: 0.0277
Epoch [1/10], Step [650/1875], Loss: 0.0255
Epoch [1/10], Step [700/1875], Loss: 0.0240
Epoch [1/10], Step [750/1875], Loss: 0.0230
Epoch [1/10], Step [800/1875], Loss: 0.0222
Epoch [1/10], Step [850/1875], Loss: 0.0216
Epoch [1/10], Step [900/1875], Loss: 0.0240
Epoch [1/10], Step [950/1875], Loss: 0.0205
Epoch [1/10], Step [1000/1875], Loss: 0.0204
Epoch [1/10], Step [1050/1875], Loss: 0.0205
Epoch [1/10], Step [1100/1875], Loss: 0.0222
Epoch [1/10], Step [1150/1875]

KeyboardInterrupt: 